# Module 2: Emotion Classification

This notebook fine-tunes a transformer emotion classifier for the mental health chatbot. It is designed for Google Colab with a T4 GPU, while the project repo keeps the reusable inference and explanation code locally.

## Why DistilBERT?

We use `distilbert-base-uncased` because it is transformer-based, lighter than BERT, fast enough for Colab T4 training, and practical for local inference. For this project, that balance is better than a larger model that may be harder to deploy or explain during assessment.

Dataset: `dair-ai/emotion` with six labels: sadness, joy, love, anger, fear, and surprise.

In [ ]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('No NVIDIA GPU detected in this runtime. Use Colab T4 for full training.')

In [ ]:
import os
import subprocess
import sys

if os.getenv('FAST_DEV_RUN', '0') == '1':
    print('Skipping package install during local FAST_DEV_RUN.')
else:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-U',
        'transformers>=4.40,<5',
        'datasets>=2.18,<5',
        'accelerate>=0.28',
        'evaluate>=0.4',
        'scikit-learn>=1.4',
    ])
# Do not upgrade pandas in Colab; Colab/GPU packages often pin pandas < 3.

## Colab Repo Setup

Run this notebook from the cloned project folder so the trained model and reports are saved back into the same structure used locally.

In [ ]:
# If you opened the notebook outside the repo in Colab, uncomment these lines once:
# !git clone https://github.com/MarwanZaineldeen/Mental-Health-Chatbot.git
# %cd Mental-Health-Chatbot

In [ ]:
import json
import os
import sys
from inspect import signature
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODEL_DIR = PROJECT_ROOT / 'src' / 'models' / 'saved_emotion_model'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'module_2_emotion_classification'
CACHE_DIR = PROJECT_ROOT / '.hf_cache'
MODEL_NAME = 'distilbert-base-uncased'
FAST_DEV_RUN = os.getenv('FAST_DEV_RUN', '0') == '1'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(PROJECT_ROOT / 'src' / 'models'))
print(f'FAST_DEV_RUN={FAST_DEV_RUN}')

In [ ]:
dataset = load_dataset('dair-ai/emotion', 'split')

if FAST_DEV_RUN:
    dataset = dataset.shuffle(seed=42)
    dataset['train'] = dataset['train'].select(range(64))
    dataset['validation'] = dataset['validation'].select(range(32))
    dataset['test'] = dataset['test'].select(range(32))

label_names = dataset['train'].features['label'].names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

print(dataset)
print(id2label)
pd.DataFrame(dataset['train'][:5])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

encoded = dataset.map(tokenize, batched=True)
encoded = encoded.rename_column('label', 'labels')
encoded = encoded.remove_columns(['text'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    cache_dir=CACHE_DIR,
)

if FAST_DEV_RUN:
    config.dim = 64
    config.hidden_dim = 128
    config.n_layers = 1
    config.n_heads = 2
    model = AutoModelForSequenceClassification.from_config(config)
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        cache_dir=CACHE_DIR,
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
    }

training_kwargs = {
    'output_dir': str(PROJECT_ROOT / 'checkpoints' / 'emotion_distilbert'),
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'num_train_epochs': 1 if FAST_DEV_RUN else 3,
    'weight_decay': 0.01,
    'save_strategy': 'no' if FAST_DEV_RUN else 'epoch',
    'load_best_model_at_end': False if FAST_DEV_RUN else True,
    'metric_for_best_model': 'macro_f1',
    'greater_is_better': True,
    'logging_steps': 1 if FAST_DEV_RUN else 50,
    'max_steps': 2 if FAST_DEV_RUN else -1,
    'report_to': 'none',
}

strategy_name = 'eval_strategy' if 'eval_strategy' in signature(TrainingArguments).parameters else 'evaluation_strategy'
training_kwargs[strategy_name] = 'epoch'

args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': encoded['train'],
    'eval_dataset': encoded['validation'],
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}

tokenizer_arg = 'processing_class' if 'processing_class' in signature(Trainer).parameters else 'tokenizer'
trainer_kwargs[tokenizer_arg] = tokenizer

trainer = Trainer(**trainer_kwargs)

In [ ]:
trainer.train()
validation_metrics = trainer.evaluate(encoded['validation'])
test_output = trainer.predict(encoded['test'])

test_predictions = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids
test_metrics = compute_metrics((test_output.predictions, test_labels))

print(validation_metrics)
print(test_metrics)

In [ ]:
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

all_label_ids = list(range(len(label_names)))
report_text = classification_report(test_labels, test_predictions, labels=all_label_ids, target_names=label_names, zero_division=0)
report_dict = classification_report(test_labels, test_predictions, labels=all_label_ids, target_names=label_names, output_dict=True, zero_division=0)
matrix = confusion_matrix(test_labels, test_predictions, labels=all_label_ids)

(REPORT_DIR / 'test_classification_report.txt').write_text(report_text, encoding='utf-8')
pd.DataFrame(report_dict).transpose().to_csv(REPORT_DIR / 'test_classification_report.csv')
pd.DataFrame(matrix, index=label_names, columns=label_names).to_csv(REPORT_DIR / 'test_confusion_matrix.csv')

summary = {
    'base_model': MODEL_NAME,
    'dataset': 'dair-ai/emotion',
    'labels': label_names,
    'validation_metrics': validation_metrics,
    'test_metrics': test_metrics,
    'explainability_method': 'word occlusion on final predicted confidence',
}
(REPORT_DIR / 'metrics_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(report_text)
print(f'Saved model to: {MODEL_DIR}')
print(f'Saved reports to: {REPORT_DIR}')

## Explainability Check

This is not a formal clinical explanation. It is a practical debugging tool: remove one word at a time and measure how much the model's confidence in the predicted emotion drops.

In [ ]:
from emotion_classifier import EmotionClassifier

classifier = EmotionClassifier(model_dir=MODEL_DIR)
examples = [
    'I feel anxious and overwhelmed and I cannot sleep.',
    'I finally feel hopeful and proud of myself.',
    'I am angry because nobody listens to me.',
]

explanations = {text: classifier.explain(text, top_k=6) for text in examples}
(REPORT_DIR / 'explanation_examples.json').write_text(json.dumps(explanations, indent=2), encoding='utf-8')
explanations